# Imports and client init

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import io
import csv
import pandas as pd
from collections import defaultdict
from dotenv import load_dotenv
from datetime import datetime
import requests
import uuid

from linalgo.hub.client import LinalgoClient
from linalgo.annotate.models import Corpus, Document, Annotation, Entity
from wsd.load_data import load_data
from lineval.utils import Body
# from linhub import models

In [ ]:
load_dotenv()
token = os.getenv('LINHUB_TOKEN')
url = "https://linhub.api.linalgo.com/v1"
client = LinalgoClient(token, url)
jack_org = "acf7a1aa-ec18-4fa2-a981-a756bc6e6af2"
test_id = "6667052e-b464-47a9-beca-dd8df8f8c632"

# Creating a Corpus (no issues)

In [ ]:
def create_corpus(corpus_id: str,
                  name: str,
                  organization: str,
                  description: str = None,
                  is_private: bool = False,
                  ) -> None:
    """ Create a corpus in the hub

    Parameters
    ----------
    corpus_id : str
        The id of the corpus
    name : str
        The name of the corpus
    description : str
        The description of the corpus
    is_private : bool
        Whether the corpus is private
    organization : str
        The id of the organization

    Returns
    -------
    None
        """
    post_url = url + f"/corpora/"
    data  = {
      "id": corpus_id,
      "name": name,
      "description": description,
      "is_private": is_private,
      "organization": organization,
    }
    client.post(url = post_url, data= data)
    pass

In [ ]:
# create_corpus(corpus_id = test_id, name = "test", organization = jack_org)

In [ ]:
corpus = client.get_corpus(test_id)
corpus.__dict__

# Creating a task (no issues)

In [ ]:
#check existing task data structure
example_task_id = 'd3ce7764-eb85-4999-b965-c028f539ee33'
existing_task = client.get_task(example_task_id, verbose= True)

In [ ]:
existing_task.__dict__.keys()

In [ ]:
existing_task.entities[0].__dict__

In [ ]:
#Generate a UID for the task
new_task_id = str(uuid.uuid4())
new_task_id

In [ ]:
def create_task(
    name: str,
    organization: str,
    task_id : str = str(uuid.uuid4()),
    description: str = None,
    entities: list[Entity] = [],
    ) -> None:


    serialized_entities = [entity.id for entity in entities]

    post_url = url + f"/tasks/"
    data  = {
        "id": task_id,
        "name": name,
        "slug": name,
        "organization": organization,
        "description": description,
        "entities": serialized_entities,
        "corpora": [corpus.id],
    }
    client.post(url = post_url, data= data)
    pass

In [ ]:
# create_task(name = "test_task",
#             organization = jack_org,
#             task_id= new_task_id,
#             entities = existing_task.entities)

In [ ]:
# new_task = client.get_task(new_task_id)

In [ ]:
# new_task.__dict__

# Load and import Semcor docs

## Loading Semcor

In [ ]:
#loading candidates
X, y = load_data(lang='fr')

k = len(X)
X_test, y_test = X[:k], y[:k]
len(X_test), len(y_test)

In [ ]:
# canidates to corpus
semcor_corpus = Corpus(name='Semcor')

grouped_X = defaultdict(list)
for i,row in enumerate(X_test):
    row.lemma_meaning = y_test[i]
    grouped_X[(row.lemma, row.pos)].append(row)
items = grouped_X.items()
docs = []
for g, cands in items:
    contexts = "\n".join([anno.context for anno in cands])
    doc = Document(content=contexts,
                   corpus=semcor_corpus
                   )
    doc_annos = []
    for c in cands:
        anno = Annotation(document=doc,
                          entity=c.lemma_meaning,
                          body=Body(text=c.text, context=c.context),
                          task="task",
                          annotator="none",
                          target={},
                          created=datetime.now())
        doc_annos.append(anno)
    doc.annotations = set(doc_annos)
    docs.append(doc)

semcor_corpus.documents = docs
X_docs = semcor_corpus.documents
len(X_docs)

In [ ]:
n=0
example_doc = X_docs[n]
example_doc.__dict__.keys()

## API call

In [ ]:
example_docs_payload = [{
  "id": example_doc.id,
  "uri": example_doc.uri,
  "title": f"doc_{n}",
  "content": example_doc.content,
  "corpus": test_id,
}]
example_docs_payload

In [ ]:
docs_payload = []
for n in range(10):
    doc = X_docs[n]
    payload = {
      "id": doc.id,
      "uri": uuid.uuid4(),
      "title": f"doc_{n}",
      "content": doc.content,
      "corpus": test_id,
    }
    docs_payload.append(payload)

In [ ]:
df = pd.DataFrame(docs_payload)
df

In [ ]:
def post(url, data=None, json=None, files = None):
    headers = {'Authorization': f"Token {token}"}
    if files:
        print("Posting files")
        res = requests.post(url, headers=headers, files=files, data=data, json=json)
    else:
        print("Posting data")
        res = requests.post(url, data=data, json=json, headers=headers)
    if 200 <= res.status_code < 300:
        return res
    if res.status_code == 401:
        raise Exception(f"Authentication failed. Please check your token.")
    elif res.status_code == 404:
        raise Exception(f"{url} not found.")
    else:
        raise Exception(
            f"Request returned status {res.status_code}, {res.content}")

In [ ]:
def post_documents(docs_payload: list[dict]) -> None:
    post_url = url + f"/documents/import_documents/"
    payload_df = pd.DataFrame(docs_payload)
    payload_df.to_csv("data/semcor_payload.csv", index=False, quoting=csv.QUOTE_ALL)
    files = {'fileKey': open('data/semcor_payload.csv','rb')}
    corpus = {'name': 'test', 'organization': jack_org}

    r = post(url = post_url, files=files, data = corpus)
    print(r.status_code)
    pass

In [ ]:
post_documents(docs_payload)

## tentative debug using code from linhub api

In [ ]:
file = open("data/semcor_payload.csv", "rb")
file_content = io.StringIO(file.read().decode('utf-8'))
reader = csv.DictReader(
            file_content, delimiter=',', quotechar='"', dialect=csv.excel)
reader

In [ ]:
for doc in reader:
    print(doc)

In [ ]:
docs = []
for doc in reader:
    doc['corpus_id'] = uuid.UUID(doc['corpus_id'])


In [ ]:
docs[0].__dict__